# LFM3-1.7B baseline: thinking ON vs OFF

Measures how much accuracy LFM3-1.7B loses when its thinking step is switched off, and how many tokens that saves.
Benchmarks: HumanEval+, MBPP+, GSM8K, IFEval.

**Before running (Kaggle right sidebar → Session options):**
1. *Accelerator*: **GPU T4 x2** (P100 does not work with vLLM).
2. *Internet*: **On** (needs a phone-verified Kaggle account).
3. Then **Run All**. Results are saved to `/kaggle/working/results` and zipped to `/kaggle/working/results.zip`.

Expected time: smoke run ~10–20 min, full run (100 per benchmark) roughly 1–3 h with vLLM, longer with the fallback backend.

## Settings

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
SMOKE_LIMIT = 20      # quick end-to-end check
FULL_LIMIT = 100      # problems per benchmark for the real baseline (None = all)
MAX_NEW_TOKENS = 8192 # same cap for both modes (fairness)
RUN_FULL = True       # set False to stop after the smoke run

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in Session options and restart."

## 2. Install
vLLM is the fast backend. If it fails to install or load, the notebook falls back to plain `transformers` (slower but works).

In [ ]:
import subprocess, sys
# vLLM first: it pins a compatible transformers version.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install:", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])
!pip install -q evalplus absl-py langdetect immutabledict nltk accelerate

In [ ]:
import nltk
for pkg in ("punkt", "punkt_tab"):
    nltk.download(pkg, quiet=True)
print("nltk ok")

## 3. Write the evaluation package
These cells write the `lfm3_instant` package (a copy of the source in this repository).

In [ ]:
import os
os.makedirs("/kaggle/working/lfm3_instant", exist_ok=True)
%cd /kaggle/working

In [ ]:
%%writefile lfm3_instant/__init__.py
"""LFM3-Instant: make LFM3-1.7B fast and reliable without thinking."""

In [ ]:
%%writefile lfm3_instant/benchmarks.py
"""Benchmarks: loading, prompts and scoring.

Coding is scored with the official evalplus harness (HumanEval+ / MBPP+), and
instruction following with Google's original IFEval checker, so numbers are
comparable with published results.
"""
from __future__ import annotations

import gzip
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

from .parsing import extract_code, extract_final_number, leading_imports, numbers_match

DATA_FILES = {
    "HumanEvalPlus.jsonl.gz": "https://github.com/evalplus/humanevalplus_release/releases/download/v0.1.10/HumanEvalPlus.jsonl.gz",
    "MbppPlus.jsonl.gz": "https://github.com/evalplus/mbppplus_release/releases/download/v0.2.0/MbppPlus.jsonl.gz",
    "gsm8k_test.jsonl": "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl",
    "ifeval_input_data.jsonl": "https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_data.jsonl",
}
IFEVAL_CODE = "https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/{}"
IFEVAL_FILES = ["instructions.py", "instructions_registry.py", "instructions_util.py", "evaluation_lib.py"]


def _download(url: str, dest: Path, attempts: int = 5) -> None:
    """Download with retries: GitHub's raw host sometimes returns 5xx / times out."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    for k in range(attempts):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f)
            tmp.replace(dest)
            return
        except Exception as e:  # noqa: BLE001 - network errors come in many types
            if k == attempts - 1:
                raise RuntimeError(f"download failed after {attempts} tries: {url}") from e
            wait = 2 ** k * 3
            print(f"  download error ({e}); retrying in {wait}s", flush=True)
            time.sleep(wait)


def _github_asset_api_url(url: str) -> str | None:
    """Map a github.com/<o>/<r>/releases/download/<tag>/<file> link to its API asset URL.

    The normal release link redirects through a host that some networks time out on;
    the API endpoint serves the same bytes by another route.
    """
    m = re.match(r"https://github\.com/([^/]+)/([^/]+)/releases/download/([^/]+)/([^/]+)$", url)
    if not m:
        return None
    owner, repo, tag, fname = m.groups()
    api = f"https://api.github.com/repos/{owner}/{repo}/releases/tags/{tag}"
    with urllib.request.urlopen(api, timeout=60) as r:
        assets = json.load(r)["assets"]
    return next((a["url"] for a in assets if a["name"] == fname), None)


def ensure_data(data_dir: Path) -> None:
    for name, url in DATA_FILES.items():
        dest = data_dir / name
        if dest.exists():
            continue
        print(f"downloading {name}")
        try:
            _download(url, dest, attempts=3)
        except RuntimeError:
            api = _github_asset_api_url(url)
            if not api:
                raise
            print(f"  retrying {name} through the GitHub API")
            req = urllib.request.Request(api, headers={"Accept": "application/octet-stream"})
            dest.parent.mkdir(parents=True, exist_ok=True)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest.with_suffix(".part"), "wb") as f:
                shutil.copyfileobj(r, f)
            dest.with_suffix(".part").replace(dest)


def ensure_ifeval_code(vendor_dir: Path) -> None:
    pkg = vendor_dir / "instruction_following_eval"
    for name in IFEVAL_FILES:
        if not (pkg / name).exists():
            _download(IFEVAL_CODE.format(name), pkg / name)
    (pkg / "__init__.py").touch()
    if str(vendor_dir) not in sys.path:
        sys.path.insert(0, str(vendor_dir))


@dataclass
class Item:
    bench: str
    id: str
    messages: list[dict]
    meta: dict = field(default_factory=dict)


def _jsonl(path: Path) -> list[dict]:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def _subset(rows: list, limit: int | None, seed: int) -> list:
    """Seeded random subset, kept in original order, so runs are reproducible and unbiased."""
    if not limit or limit >= len(rows):
        return rows
    keep = set(random.Random(seed).sample(range(len(rows)), limit))
    return [r for i, r in enumerate(rows) if i in keep]


class Benchmark:
    name: str
    primary: str  # key of the headline metric in per-item scores
    sampling: str  # "general" or "math" (Liquid's recommended presets)

    def load(self, data_dir: Path, limit: int | None, seed: int) -> list[Item]: ...

    def score(self, items: list[Item], answers: list[str], work_dir: Path) -> list[dict]: ...


# ---------------------------------------------------------------- coding (evalplus)

CODE_INSTRUCTION = (
    "Return the complete function implementation, including the signature and any "
    "imports it needs, in a single ```python code block."
)


class _EvalPlus(Benchmark):
    primary = "plus_pass"
    sampling = "general"
    dataset: str  # evalplus dataset name
    file: str
    override_env: str

    def _rows(self, data_dir: Path) -> list[dict]:
        return _jsonl(data_dir / self.file)

    def score(self, items, answers, work_dir):
        data_dir = Path(items[0].meta["data_dir"])
        work_dir.mkdir(parents=True, exist_ok=True)
        by_id = {it.id: (it, a) for it, a in zip(items, answers)}
        samples = work_dir / f"{self.name}_samples.jsonl"
        with open(samples, "w") as f:
            # evalplus requires every problem; unselected ones get an empty solution and are ignored below.
            for row in self._rows(data_dir):
                sol = ""
                if row["task_id"] in by_id:
                    it, ans = by_id[row["task_id"]]
                    sol = self.to_solution(it, ans)
                f.write(json.dumps({"task_id": row["task_id"], "solution": sol}) + "\n")
        results = work_dir / f"{self.name}_samples_eval_results.json"
        results.unlink(missing_ok=True)  # evalplus silently reuses stale results otherwise
        env = {**os.environ, self.override_env: str((data_dir / self.file).resolve())}
        # Use the running interpreter: a PATH lookup can find another Python's evalplus.
        cmd = [sys.executable, "-m", "evalplus.evaluate"] + [
            self.dataset, "--samples", str(samples), "--parallel", str(max(1, (os.cpu_count() or 2))),
        ]
        proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if proc.returncode != 0 or not results.exists():
            raise RuntimeError(f"evalplus failed:\n{proc.stdout[-2000:]}\n{proc.stderr[-2000:]}")
        ev = json.load(open(results))["eval"]
        out = []
        for it in items:
            r = ev[it.id][0]
            base = r["base_status"] == "pass"
            out.append({"base_pass": base, "plus_pass": base and r["plus_status"] == "pass"})
        return out


class HumanEvalPlus(_EvalPlus):
    name, dataset, file, override_env = "humaneval", "humaneval", "HumanEvalPlus.jsonl.gz", "HUMANEVAL_OVERRIDE_PATH"

    def load(self, data_dir, limit, seed):
        items = []
        for r in _subset(self._rows(data_dir), limit, seed):
            prompt = f"Complete the following Python function.\n\n```python\n{r['prompt'].rstrip()}\n```\n\n{CODE_INSTRUCTION}"
            items.append(Item(self.name, r["task_id"], [{"role": "user", "content": prompt}],
                              {"entry_point": r["entry_point"], "prompt": r["prompt"], "data_dir": str(data_dir)}))
        return items

    def to_solution(self, it, answer):
        code = extract_code(answer, it.meta["entry_point"])
        if f"def {it.meta['entry_point']}" not in code:  # model wrote only the body
            return it.meta["prompt"] + code
        return leading_imports(it.meta["prompt"]) + "\n\n" + code


class MbppPlus(_EvalPlus):
    name, dataset, file, override_env = "mbpp", "mbpp", "MbppPlus.jsonl.gz", "MBPP_OVERRIDE_PATH"

    def load(self, data_dir, limit, seed):
        items = []
        for r in _subset(self._rows(data_dir), limit, seed):
            task = r["prompt"].strip().strip('"').strip()
            prompt = (f"{task}\n\nThe function must be named `{r['entry_point']}` and pass the test above.\n\n"
                      f"{CODE_INSTRUCTION}")
            items.append(Item(self.name, r["task_id"], [{"role": "user", "content": prompt}],
                              {"entry_point": r["entry_point"], "data_dir": str(data_dir)}))
        return items

    def to_solution(self, it, answer):
        return extract_code(answer, it.meta["entry_point"])


# ---------------------------------------------------------------- math

class GSM8K(Benchmark):
    name, primary, sampling = "gsm8k", "correct", "math"

    def load(self, data_dir, limit, seed):
        rows = _jsonl(data_dir / "gsm8k_test.jsonl")
        items = []
        for i, r in enumerate(rows):
            gold = float(r["answer"].rsplit("####", 1)[-1].strip().replace(",", ""))
            prompt = f"{r['question']}\n\nSolve the problem and put the final numeric answer inside \\boxed{{}}."
            items.append(Item(self.name, f"gsm8k/{i}", [{"role": "user", "content": prompt}], {"gold": gold}))
        return _subset(items, limit, seed)

    def score(self, items, answers, work_dir):
        out = []
        for it, a in zip(items, answers):
            pred = extract_final_number(a)
            out.append({"correct": numbers_match(pred, it.meta["gold"]), "pred": pred})
        return out


# ---------------------------------------------------------------- instruction following

class IFEval(Benchmark):
    name, primary, sampling = "ifeval", "prompt_strict", "general"

    def __init__(self, vendor_dir: Path):
        self.vendor_dir = vendor_dir

    def load(self, data_dir, limit, seed):
        rows = _jsonl(data_dir / "ifeval_input_data.jsonl")
        items = [Item(self.name, f"ifeval/{r['key']}", [{"role": "user", "content": r["prompt"]}], {"row": r})
                 for r in rows]
        return _subset(items, limit, seed)

    def score(self, items, answers, work_dir):
        ensure_ifeval_code(self.vendor_dir)
        from instruction_following_eval import evaluation_lib as ev

        out = []
        for it, a in zip(items, answers):
            r = it.meta["row"]
            inp = ev.InputExample(key=r["key"], instruction_id_list=r["instruction_id_list"],
                                  prompt=r["prompt"], kwargs=r["kwargs"])
            strict = ev.test_instruction_following_strict(inp, {r["prompt"]: a})
            loose = ev.test_instruction_following_loose(inp, {r["prompt"]: a})
            out.append({
                "prompt_strict": strict.follow_all_instructions,
                "prompt_loose": loose.follow_all_instructions,
                "inst_strict": sum(strict.follow_instruction_list) / len(strict.follow_instruction_list),
            })
        return out


def get_benchmarks(names: list[str], vendor_dir: Path) -> list[Benchmark]:
    table = {"humaneval": HumanEvalPlus, "mbpp": MbppPlus, "gsm8k": GSM8K}
    out = []
    for n in names:
        if n == "ifeval":
            out.append(IFEval(vendor_dir))
        elif n in table:
            out.append(table[n]())
        else:
            raise ValueError(f"unknown benchmark {n!r}; choose from humaneval, mbpp, gsm8k, ifeval")
    return out

In [ ]:
%%writefile lfm3_instant/generate.py
"""Generation backends.

- vllm: fast batched generation on a GPU (use this on Kaggle).
- hf:   plain transformers, slower; a fallback if vLLM will not install or load.
- mock: no model; replays fixed answers so the whole pipeline can be tested on a CPU.
"""
from __future__ import annotations

import time
from dataclasses import dataclass

# Liquid's recommended sampling for LFM3 (model card).
SAMPLING_PRESETS = {
    "general": {"temperature": 0.1, "top_k": 50, "top_p": 1.0, "repetition_penalty": 1.1},
    "math": {"temperature": 1.0, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0},
}


@dataclass
class Generation:
    text: str
    n_tokens: int
    finish_reason: str  # "stop" or "length"


class Backend:
    tokenizer = None

    def render(self, messages: list[dict], thinking: bool) -> str:
        """Apply the model's own chat template; enable_thinking switches the mode."""
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=thinking
        )

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=False)) if text else 0

    def generate(self, prompts: list[str], sampling: dict, max_new_tokens: int, seed: int) -> list[Generation]:
        raise NotImplementedError

    def _strip_special(self, text: str) -> str:
        # Keep <think> tags (they may be special tokens) but drop end-of-turn / padding markers.
        for tok in getattr(self.tokenizer, "all_special_tokens", []):
            if tok and tok not in ("<think>", "</think>"):
                text = text.replace(tok, "")
        return text


class VLLMBackend(Backend):
    def __init__(self, model: str, dtype: str = "auto", max_model_len: int = 16384,
                 gpu_memory_utilization: float = 0.90, tensor_parallel_size: int = 1,
                 enforce_eager: bool = False, seed: int = 0):
        import torch
        from vllm import LLM

        if dtype == "auto":
            # Kaggle's T4 / P100 have no bfloat16; fall back to float16.
            dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
        self.llm = LLM(model=model, dtype=dtype, max_model_len=max_model_len,
                       gpu_memory_utilization=gpu_memory_utilization,
                       tensor_parallel_size=tensor_parallel_size, enforce_eager=enforce_eager, seed=seed)
        self.tokenizer = self.llm.get_tokenizer()
        self.dtype = dtype

    def generate(self, prompts, sampling, max_new_tokens, seed):
        from vllm import SamplingParams

        params = SamplingParams(max_tokens=max_new_tokens, seed=seed, skip_special_tokens=False, **sampling)
        outs = self.llm.generate(prompts, params, use_tqdm=True)
        return [Generation(self._strip_special(o.outputs[0].text), len(o.outputs[0].token_ids),
                           o.outputs[0].finish_reason or "stop") for o in outs]


class HFBackend(Backend):
    def __init__(self, model: str, dtype: str = "auto", batch_size: int = 8, seed: int = 0):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model, padding_side="left")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if dtype == "auto":
            dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
                "float16" if torch.cuda.is_available() else "float32")
        self.dtype = dtype
        self.model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=getattr(torch, dtype), device_map="auto")
        self.model.eval()
        self.batch_size = batch_size

    def generate(self, prompts, sampling, max_new_tokens, seed):
        torch = self.torch
        torch.manual_seed(seed)
        order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))  # similar lengths per batch
        results: list[Generation | None] = [None] * len(prompts)
        eos = self.tokenizer.eos_token_id
        for start in range(0, len(order), self.batch_size):
            idx = order[start : start + self.batch_size]
            batch = self.tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                                   add_special_tokens=False).to(self.model.device)
            do_sample = sampling["temperature"] > 0
            with torch.no_grad():
                out = self.model.generate(
                    **batch, max_new_tokens=max_new_tokens, do_sample=do_sample,
                    temperature=sampling["temperature"] if do_sample else None,
                    top_k=sampling["top_k"] if do_sample else None,
                    top_p=sampling["top_p"] if do_sample else None,
                    repetition_penalty=sampling["repetition_penalty"],
                    pad_token_id=self.tokenizer.pad_token_id,
                )
            new = out[:, batch["input_ids"].shape[1]:]
            for row, i in zip(new.tolist(), idx):
                stop = next((k for k, t in enumerate(row) if t == eos), None)
                toks = row if stop is None else row[:stop]
                text = self._strip_special(self.tokenizer.decode(toks, skip_special_tokens=False))
                results[i] = Generation(text, len(toks), "length" if stop is None and len(row) >= max_new_tokens else "stop")
            print(f"  hf batch {start // self.batch_size + 1}/{-(-len(order) // self.batch_size)}", flush=True)
        return results  # type: ignore[return-value]


class MockBackend(Backend):
    """Replays answers keyed by user message. Used to test parsing and scoring end to end."""

    def __init__(self, answers: dict[str, str], reasoning: str = "Let me think step by step about this."):
        self.answers, self.reasoning = answers, reasoning

    def render(self, messages, thinking):
        return ("<think>\n" if thinking else "<think></think>") + "\x00" + messages[-1]["content"]

    def count_tokens(self, text):
        return len(text.split())

    def generate(self, prompts, sampling, max_new_tokens, seed):
        out = []
        for p in prompts:
            thinking = p.startswith("<think>\n")
            ans = self.answers.get(p.split("\x00", 1)[1], "")
            text = f"{self.reasoning}\n</think>\n\n{ans}" if thinking else ans
            out.append(Generation(text, self.count_tokens(text), "stop"))
        return out


def make_backend(name: str, model: str, **kw) -> Backend:
    if name == "vllm":
        return VLLMBackend(model, **{k: v for k, v in kw.items() if k in (
            "dtype", "max_model_len", "gpu_memory_utilization", "tensor_parallel_size", "enforce_eager", "seed")})
    if name == "hf":
        return HFBackend(model, **{k: v for k, v in kw.items() if k in ("dtype", "batch_size", "seed")})
    raise ValueError(f"unknown backend {name!r}")


class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.t0

In [ ]:
%%writefile lfm3_instant/parsing.py
"""Parse raw model outputs: split thinking from the answer, extract code and numbers."""
from __future__ import annotations

import re
from dataclasses import dataclass

THINK_OPEN = "<think>"
THINK_CLOSE = "</think>"


@dataclass
class SplitOutput:
    reasoning: str
    answer: str
    status: str  # "closed" (thinking finished), "unclosed" (thinking never finished), "none" (no thinking)


def prompt_opens_thinking(rendered_prompt: str) -> bool:
    """True if the chat template already opened a <think> block that the model must close."""
    tail = rendered_prompt.rstrip()
    last_open = tail.rfind(THINK_OPEN)
    return last_open != -1 and tail.find(THINK_CLOSE, last_open) == -1


def split_thinking(raw: str, prompt_opened_think: bool = False) -> SplitOutput:
    """Separate the reasoning trace from the final answer.

    If the trace never closes (truncated or runaway thinking), the answer is empty,
    so the sample is scored as a failure rather than graded on half-finished reasoning.
    """
    if THINK_CLOSE in raw:
        pre, post = raw.rsplit(THINK_CLOSE, 1)
        return SplitOutput(pre.replace(THINK_OPEN, "").strip(), post.strip(), "closed")
    if THINK_OPEN in raw or prompt_opened_think:
        return SplitOutput(raw.split(THINK_OPEN, 1)[-1].strip(), "", "unclosed")
    return SplitOutput("", raw.strip(), "none")


def leaked_thinking(raw: str) -> bool:
    """In no-thinking mode, did the model emit thinking tags anyway?"""
    return THINK_OPEN in raw or THINK_CLOSE in raw


_FENCE = re.compile(r"```[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)```", re.DOTALL)


def extract_code(answer: str, entry_point: str | None = None) -> str:
    """Pull Python code out of a chat answer.

    Prefers the last fenced block that defines `entry_point`, then the longest
    python block, then the whole answer if there are no fences.
    """
    blocks = [(lang.lower(), body) for lang, body in _FENCE.findall(answer)]
    if not blocks:
        # An unterminated fence (e.g. answer cut off) still often holds usable code.
        m = re.search(r"```[ \t]*[A-Za-z0-9_+-]*[^\n]*\n(.*)", answer, re.DOTALL)
        return (m.group(1) if m else answer).strip("\n")
    py = [b for lang, b in blocks if lang in ("", "python", "py", "python3")] or [b for _, b in blocks]
    if entry_point:
        defining = [b for b in py if re.search(rf"\bdef\s+{re.escape(entry_point)}\s*\(", b)]
        if defining:
            return defining[-1].strip("\n")
    return max(py, key=len).strip("\n")


def leading_imports(code: str) -> str:
    """Import lines from the top-level of a snippet (used to carry HumanEval prompt imports)."""
    return "\n".join(
        line for line in code.splitlines() if re.match(r"^(import|from)\s+\S+", line)
    )


_NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def _boxed_contents(text: str) -> list[str]:
    """Contents of every \\boxed{...}, handling nested braces."""
    out, i = [], 0
    while True:
        i = text.find("\\boxed{", i)
        if i == -1:
            return out
        j, depth = i + len("\\boxed{"), 1
        start = j
        while j < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[j], 0)
            j += 1
        out.append(text[start : j - 1] if depth == 0 else text[start:])
        i = j


def extract_final_number(text: str) -> float | None:
    """Final numeric answer: \\boxed{} first, then '####', then 'answer is', then last number."""
    boxed = _boxed_contents(text)
    if boxed:
        candidate = boxed[-1]
    elif "####" in text:
        candidate = text.rsplit("####", 1)[-1]
    else:
        m = re.findall(r"answer\s*(?:is|:)\s*([^\n]+)", text, flags=re.IGNORECASE)
        candidate = m[-1] if m else text
    nums = _NUM.findall(candidate) or _NUM.findall(text)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", ""))
    except ValueError:
        return None


def numbers_match(pred: float | None, gold: float, tol: float = 1e-4) -> bool:
    return pred is not None and abs(pred - gold) <= tol * max(1.0, abs(gold))

In [ ]:
%%writefile lfm3_instant/run_eval.py
"""Baseline evaluation: LFM3-1.7B with thinking ON vs OFF.

Example (Kaggle, GPU):
    python -m lfm3_instant.run_eval --model LiquidAI/LFM3-1.7B --limit 100 --out results/baseline
"""
from __future__ import annotations

import argparse
import json
import statistics
from pathlib import Path

from .benchmarks import ensure_data, get_benchmarks
from .generate import SAMPLING_PRESETS, Backend, Timer, make_backend
from .parsing import leaked_thinking, prompt_opens_thinking, split_thinking

MODES = {"think": True, "nothink": False}


def check_template(backend: Backend, messages: list[dict], out_dir: Path) -> bool:
    """Render one prompt in both modes and save them. Returns False if the switch does nothing."""
    on, off = backend.render(messages, True), backend.render(messages, False)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "template_check.txt").write_text(
        f"===== enable_thinking=True =====\n{on}\n\n===== enable_thinking=False =====\n{off}\n")
    if on == off:
        print("WARNING: the chat template renders identical prompts for thinking on/off; "
              "enable_thinking is being ignored. Results for the two modes will not differ.")
    return on != off


def _load_cache(path: Path, ids: list[str]) -> list[dict] | None:
    if not path.exists():
        return None
    rows = [json.loads(l) for l in open(path)]
    return rows if [r["id"] for r in rows] == ids else None


def run(args) -> dict:
    out = Path(args.out)
    data_dir, vendor_dir = Path(args.data_dir), Path(args.vendor_dir)
    ensure_data(data_dir)
    benches = get_benchmarks(args.benchmarks, vendor_dir)
    items = {b.name: b.load(data_dir, args.limit, args.seed) for b in benches}

    backend = args.backend_obj if getattr(args, "backend_obj", None) else make_backend(
        args.backend, args.model, dtype=args.dtype, max_model_len=args.max_model_len,
        gpu_memory_utilization=args.gpu_memory_utilization, tensor_parallel_size=args.tp,
        enforce_eager=args.enforce_eager, batch_size=args.batch_size, seed=args.seed)
    template_ok = check_template(backend, items[benches[0].name][0].messages, out)

    summary = {"model": args.model, "limit": args.limit, "seed": args.seed,
               "max_new_tokens": args.max_new_tokens, "template_switch_works": template_ok, "rows": []}
    for mode in args.modes:
        thinking = MODES[mode]
        for b in benches:
            its = items[b.name]
            mdir = out / mode
            mdir.mkdir(parents=True, exist_ok=True)
            gen_path = mdir / f"{b.name}.gen.jsonl"
            sampling = SAMPLING_PRESETS[b.sampling]
            prompts = [backend.render(it.messages, thinking) for it in its]

            cached = _load_cache(gen_path, [it.id for it in its])
            if cached is None:
                print(f"\n== generating {b.name} [{mode}] n={len(its)}", flush=True)
                with Timer() as t:
                    gens = backend.generate(prompts, sampling, args.max_new_tokens, args.seed)
                with open(gen_path, "w") as f:
                    for it, g in zip(its, gens):
                        f.write(json.dumps({"id": it.id, "text": g.text, "n_tokens": g.n_tokens,
                                            "finish_reason": g.finish_reason, "gen_seconds": t.seconds / len(its)}) + "\n")
                cached = _load_cache(gen_path, [it.id for it in its])
            else:
                print(f"\n== reusing cached generations for {b.name} [{mode}]")

            records = []
            for it, p, g in zip(its, prompts, cached):
                sp = split_thinking(g["text"], prompt_opens_thinking(p))
                records.append({
                    "id": it.id, "bench": b.name, "mode": mode, "raw": g["text"],
                    "reasoning": sp.reasoning, "answer": sp.answer, "think_status": sp.status,
                    "n_tokens": g["n_tokens"], "n_reasoning_tokens": backend.count_tokens(sp.reasoning),
                    "truncated": g["finish_reason"] == "length",
                    "leak": (not thinking) and leaked_thinking(g["text"]),
                    "gen_seconds": g["gen_seconds"],
                })
            scores = b.score(its, [r["answer"] for r in records], mdir / "scoring")
            for r, s in zip(records, scores):
                r["score"] = s
            with open(mdir / f"{b.name}.jsonl", "w") as f:
                for r in records:
                    f.write(json.dumps(r, default=str) + "\n")

            n = len(records)
            toks = [r["n_tokens"] for r in records]
            row = {
                "mode": mode, "bench": b.name, "n": n, "metric": b.primary,
                "accuracy": sum(bool(r["score"][b.primary]) for r in records) / n,
                "mean_tokens": statistics.mean(toks), "median_tokens": statistics.median(toks),
                "mean_reasoning_tokens": statistics.mean(r["n_reasoning_tokens"] for r in records),
                "truncated_rate": sum(r["truncated"] for r in records) / n,
                "unclosed_think_rate": sum(r["think_status"] == "unclosed" for r in records) / n,
                "leak_rate": sum(r["leak"] for r in records) / n,
                "gen_seconds": sum(r["gen_seconds"] for r in records),
            }
            for extra in ("base_pass", "prompt_loose", "inst_strict"):
                if extra in records[0]["score"]:
                    row[extra] = sum(float(r["score"][extra]) for r in records) / n
            summary["rows"].append(row)
            print(f"   {b.name} [{mode}] {b.primary}={row['accuracy']:.3f}  mean tokens={row['mean_tokens']:.0f}")

    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    (out / "summary.md").write_text(to_markdown(summary))
    print("\n" + to_markdown(summary))
    return summary


def to_markdown(summary: dict) -> str:
    rows = summary["rows"]
    lines = [f"# Baseline: {summary['model']}  (limit={summary['limit']}, seed={summary['seed']}, "
             f"max_new_tokens={summary['max_new_tokens']})", ""]
    if not summary["template_switch_works"]:
        lines += ["**WARNING: enable_thinking had no effect on the prompt; modes are identical.**", ""]
    lines += ["| benchmark | mode | n | score | mean tokens | median tokens | reasoning tokens | truncated | leaked thinking | gen time (s) |",
              "|---|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        lines.append(f"| {r['bench']} ({r['metric']}) | {r['mode']} | {r['n']} | {r['accuracy']:.1%} | "
                     f"{r['mean_tokens']:.0f} | {r['median_tokens']:.0f} | {r['mean_reasoning_tokens']:.0f} | "
                     f"{r['truncated_rate']:.1%} | {r['leak_rate']:.1%} | {r['gen_seconds']:.0f} |")
    by = {(r["bench"], r["mode"]): r for r in rows}
    pairs = [b for b in dict.fromkeys(r["bench"] for r in rows) if (b, "think") in by and (b, "nothink") in by]
    if pairs:
        lines += ["", "## Thinking ON vs OFF", "",
                  "| benchmark | score ON | score OFF | gap (points) | tokens ON | tokens OFF | token reduction |",
                  "|---|---|---|---|---|---|---|"]
        for b in pairs:
            on, off = by[(b, "think")], by[(b, "nothink")]
            red = f"{on['mean_tokens'] / off['mean_tokens']:.1f}x" if off["mean_tokens"] else "n/a"
            lines.append(f"| {b} | {on['accuracy']:.1%} | {off['accuracy']:.1%} | "
                         f"{(on['accuracy'] - off['accuracy']) * 100:+.1f} | {on['mean_tokens']:.0f} | "
                         f"{off['mean_tokens']:.0f} | {red} |")
        lines += ["", "LFM3-Instant's goal: keep the OFF token count while closing the score gap."]
    return "\n".join(lines) + "\n"


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", default="LiquidAI/LFM3-1.7B")
    p.add_argument("--backend", default="vllm", choices=["vllm", "hf"])
    p.add_argument("--modes", nargs="+", default=["think", "nothink"], choices=list(MODES))
    p.add_argument("--benchmarks", nargs="+", default=["humaneval", "mbpp", "gsm8k", "ifeval"])
    p.add_argument("--limit", type=int, default=None, help="problems per benchmark (seeded random subset)")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max-new-tokens", type=int, default=8192, help="same cap for both modes, for fairness")
    p.add_argument("--out", default="results/baseline")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--vendor-dir", default="vendor")
    p.add_argument("--dtype", default="auto")
    p.add_argument("--max-model-len", type=int, default=16384)
    p.add_argument("--gpu-memory-utilization", type=float, default=0.90)
    p.add_argument("--tp", type=int, default=1, help="tensor parallel GPUs (2 on Kaggle's T4 x2)")
    p.add_argument("--enforce-eager", action="store_true", help="try this if vLLM fails to capture CUDA graphs")
    p.add_argument("--batch-size", type=int, default=8, help="hf backend only")
    return p.parse_args(argv)


if __name__ == "__main__":
    run(parse_args())

## 4. Template check
Shows the exact prompt the model sees in each mode. The two must differ (OFF should contain an empty `<think></think>`).

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)
except Exception as e:  # LFM3 is new; an older transformers may not know it
    print("tokenizer failed (", e, ") -> upgrading transformers; if this happens, RESTART the kernel and Run All again")
    !pip install -q -U transformers
    raise
msgs = [{"role": "user", "content": "What is 2+2?"}]
on = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
off = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print("===== thinking ON =====\n" + on)
print("===== thinking OFF =====\n" + off)
print("Switch works:", on != off)

## 5. Choose backend
Tries vLLM on one small generation; if that fails, uses `transformers`.

In [ ]:
import subprocess, sys
test = '''
from vllm import LLM, SamplingParams
llm = LLM(model="%s", dtype="float16", max_model_len=4096, gpu_memory_utilization=0.85)
print(llm.generate(["Hello"], SamplingParams(max_tokens=8))[0].outputs[0].text)
''' % MODEL
r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
BACKEND = "vllm" if r.returncode == 0 else "hf"
print("Backend:", BACKEND)
if BACKEND == "hf":
    print("vLLM failed, last lines:\n", r.stderr[-2000:])

## 6. Smoke run (20 problems per benchmark)
Checks everything end to end before spending hours.

In [ ]:
!python -m lfm3_instant.run_eval --model {MODEL} --backend {BACKEND} --limit {SMOKE_LIMIT} --max-new-tokens {MAX_NEW_TOKENS} --out results/smoke

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/smoke/summary.md").read()))
print(open("results/smoke/template_check.txt").read()[:3000])

## 7. Full baseline run
Generations are cached per benchmark/mode, so re-running this cell after a disconnect resumes instead of starting over (as long as `/kaggle/working` still exists).

In [ ]:
if RUN_FULL:
    limit = "" if FULL_LIMIT is None else f"--limit {FULL_LIMIT}"
    !python -m lfm3_instant.run_eval --model {MODEL} --backend {BACKEND} {limit} --max-new-tokens {MAX_NEW_TOKENS} --out results/baseline

In [ ]:
if RUN_FULL:
    display(Markdown(open("results/baseline/summary.md").read()))

## 8. Save results
Download `results.zip` from the Output panel (or commit the notebook with *Save Version → Save & Run All* to keep outputs).

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "/kaggle/working/results")
print("saved /kaggle/working/results.zip")